# Lab 3: NumPy Vectorization and Broadcasting

 Name:**Mainza Muya**   
 Student number:**202406802**   
 Course:**ICD341 Advanced Statistical Computing**  

In [1]:
import numpy as np

## Part A: Basic vectorization

In [2]:
x = np.array([1, 2, 3, 4, 5])

# loop version
result = []
for value in x:
    result.append(value ** 2)
print(result)

# vectorized version
print(x ** 2)

[np.int64(1), np.int64(4), np.int64(9), np.int64(16), np.int64(25)]
[ 1  4  9 16 25]


In [3]:
# Task 1: y = 2x + 5
y = 2*x + 5
print(y)

[ 7  9 11 13 15]


**Questions**
1. **Vectorization** means applying an operation to a whole array at once instead of element by element in a Python loop.
2. It is useful because it is faster, shorter and easier to read.
3. A **Python loop** runs interpreted code with overhead on every element. A **vectorized operation** runs a single pre-compiled C loop over contiguous, typed memory.

## Part B: Vectorized statistical transformation

In [4]:
marks = np.array([40, 55, 62, 70, 81, 90])

adjusted = marks + 5
print(adjusted)

percentage = marks / 100
print(percentage)

[45 60 67 75 86 95]
[0.4  0.55 0.62 0.7  0.81 0.9 ]


In [5]:
# Task 2: standardization
z = (marks - np.mean(marks)) / np.std(marks)
print(z)
print("Mean of z:", z.mean())
print("Approximately zero?", np.isclose(z.mean(), 0))

[-1.59799895 -0.68774638 -0.26296185  0.22250618  0.89002473  1.43617627]
Mean of z: 2.590520390792032e-16
Approximately zero? True


The mean of `z` is essentially zero (of the order of 1e-16, due to floating-point rounding).

_Note:_ `np.std` uses the population formula (`ddof=0`). The sample standard deviation *s* in the formula would use `np.std(marks, ddof=1)`.

## Part C: Broadcasting

In [6]:
data = np.array([
    [60, 70, 80],
    [50, 65, 75],
    [90, 85, 95]
])
adjustment = np.array([5, 0, 2])

result = data + adjustment
print(result)
print(data.shape, adjustment.shape)

[[65 70 82]
 [55 65 77]
 [95 85 97]]
(3, 3) (3,)


**Why does this work?** NumPy's broadcasting rules compare shapes from the rightmost dimension. `adjustment` has shape `(3,)`, which is treated as `(1, 3)`. A dimension of size 1 is virtually stretched to match the other array, so the adjustment vector is applied to each of the 3 rows. The data is not actually copied.

## Part D: Broadcasting with statistics

In [7]:
data = np.array([
    [10, 20, 30],
    [20, 30, 40],
    [30, 40, 50]
])

means = data.mean(axis=0)
print("Column means:", means)

centered = data - means
print(centered)

# Task 3
print(centered.mean(axis=0))

Column means: [20. 30. 40.]
[[-10. -10. -10.]
 [  0.   0.   0.]
 [ 10.  10.  10.]]
[0. 0. 0.]


**Why is subtracting the vector of means from the entire matrix possible?** `means` has shape `(3,)` and `data` has shape `(3, 3)`. Broadcasting stretches `means` across every row, so each column has its own mean subtracted. The column means of `centered` are therefore `[0. 0. 0.]`.

## Part E: Vectorization versus loops

In [8]:
x = np.arange(1_000_000)

# loop
result_loop = []
for value in x:
    result_loop.append(value**2 + 2*value + 1)

# vectorized
result_vectorized = x**2 + 2*x + 1

np.allclose(result_loop, result_vectorized)

True

**Questions**
1. **Are the results equal?** Yes, `np.allclose` returns `True`.
2. **Easier to read?** The vectorized version: one line that matches the mathematical formula.
3. **Which is faster?** The vectorized version.
4. **Why?** The loop pays Python interpreter overhead on each of 1,000,000 iterations (type checks, object creation, `append`). NumPy performs the arithmetic in compiled C over contiguous memory, often using SIMD instructions.

## Part F: Statistical computing application

In [9]:
np.random.seed(123)
x = np.random.normal(100, 15, 100_000)

z = (x - x.mean()) / x.std()

print(z.mean())
print(z.std())

-9.839595804805867e-16
0.9999999999999999


**Interpretation:** `z.mean()` is essentially 0 and `z.std()` is 1, as standardization requires. Vectorization is particularly useful for large statistical datasets because computation time grows with the sample size. A loop over 100,000 (or millions of) values is slow, whereas vectorized code stays fast and concise. Simulation, bootstrap and MCMC methods later in the course rely on this.

## Lab 3 Challenge

In [11]:
def standardize(x, axis=0):
    """Standardize a NumPy array.
    1-D input: standardized as a whole.
    2-D input: standardized column by column (axis=0)."""
    x = np.asarray(x, dtype=float)
    mean = np.mean(x, axis=axis, keepdims=True)
    std = np.std(x, axis=axis, keepdims=True)
    return (x - mean) / std

In [12]:
# Test 1: small 1-D array
a1 = np.array([1, 2, 3, 4, 5])
z1 = standardize(a1)
print(z1, z1.mean(), z1.std())

# Test 2: large random 1-D array
np.random.seed(1)
a2 = np.random.normal(10, 3, 1000)
z2 = standardize(a2)
print(z2.mean(), z2.std())

# Test 3: 2-D array, column by column
a3 = np.array([[1, 10, 100], [2, 20, 200], [3, 30, 300]])
z3 = standardize(a3)
print(z3)
print(z3.mean(axis=0), z3.std(axis=0))

[-1.41421356 -0.70710678  0.          0.70710678  1.41421356] 0.0 0.9999999999999999
-3.1974423109204506e-17 1.0
[[-1.22474487 -1.22474487 -1.22474487]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  1.22474487]]
[0. 0. 0.] [1. 1. 1.]


`keepdims=True` preserves the reduced axis so that broadcasting works. For 1-D input the function behaves exactly like the original one, and for 2-D input it standardizes each column separately.